In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install -q langchain-text-splitters
!pip install -q weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.2/643.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.0 MB/s eta 0:00:00


In [ ]:
import time
import json
import torch
import numpy as np
import pandas as pd
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery
from tqdm.notebook import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# jinna5-nano

# Weaviate

## config

In [ ]:
# ===================== CONFIG =====================
corpus_file   = "/content/long_docs.jsonl"
chunk_size    = 600
chunk_overlap = 90
batch_size    = 64
model_name    = "jinaai/jina-embeddings-v5-text-nano-retrieval"
qrels_file    = "/content/qrels.miracl-v1.0-fa-dev.tsv"
topics_file   = "/content/topics.miracl-v1.0-fa-dev.tsv"
collection    = "Miracl"

In [ ]:

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks_text = []
all_chunks_meta = []
total_docs = 0

print("Chunking documents...")
with open(corpus_file, 'r', encoding='utf-8') as f:
    for doc_idx, line in enumerate(tqdm(f, desc="Chunking")):
        if not line.strip():
            continue

        doc    = json.loads(line)
        text   = doc.get("text", "")
        doc_id = str(doc.get("docid") or doc.get("id") or doc_idx)

        if len(text) < 600:
            continue

        total_docs += 1
        chunks = splitter.split_text(text)

        for chunk_idx, chunk_text in enumerate(chunks):
            all_chunks_text.append(chunk_text)
            all_chunks_meta.append({
                "doc_id":              doc_id,
                "chunk_id":            chunk_idx,
                "original_doc_length": len(text)
            })

print(f"Total docs:   {total_docs}")
print(f"Total chunks: {len(all_chunks_text)}\n")

# ===================== ENCODE =====================
print(f"Encoding {len(all_chunks_text)} chunks with batch_size={batch_size}...")
encode_start = time.time()

embeddings = model.encode(
    all_chunks_text,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')

encode_elapsed = time.time() - encode_start

print(f"""
============== ENCODING ==============
  Total time:        {encode_elapsed:.1f}s ({encode_elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks_text)/encode_elapsed:.1f}
  Embedding dim:     {embeddings.shape[1]}
  Matrix size:       {embeddings.nbytes/1024**2:.2f} MB
======================================
""")

# ===================== BUILD WEAVIATE INDEX =====================
print("Building Weaviate index...")
index_start = time.time()

# Embedded mode — no server needed, like Qdrant in-memory
client = weaviate.connect_to_embedded()

# Delete collection if exists
if client.collections.exists(collection):
    client.collections.delete(collection)
    print(f"Deleted existing collection: {collection}")

# Create collection
client.collections.create(
    name=collection,
    vectorizer_config=Configure.Vectorizer.none(),   # we provide our own vectors
    properties=[
        Property(name="doc_id",              data_type=DataType.TEXT),
        Property(name="chunk_text",          data_type=DataType.TEXT),
        Property(name="chunk_id",            data_type=DataType.INT),
        Property(name="original_doc_length", data_type=DataType.INT),
    ]
)
print(f"Created collection: {collection}")

# Upload in batches
col = client.collections.get(collection)
upload_batch = 512

with col.batch.fixed_size(batch_size=upload_batch) as batch:
    for i in tqdm(range(len(embeddings)), desc="Uploading to Weaviate"):
        batch.add_object(
            properties={
                "doc_id":              all_chunks_meta[i]["doc_id"],
                "chunk_text":          all_chunks_text[i],
                "chunk_id":            all_chunks_meta[i]["chunk_id"],
                "original_doc_length": all_chunks_meta[i]["original_doc_length"],
            },
            vector=embeddings[i].tolist()
        )

index_elapsed = time.time() - index_start

print(f"""
============== WEAVIATE INDEX ==============
  Collection:    {collection}
  Total vectors: {col.aggregate.over_all(total_count=True).total_count}
  Build time:    {index_elapsed:.2f}s
============================================
""")

# ===================== LOAD TOPICS & QRELS =====================
print("Loading filtered doc IDs...")
filtered_doc_ids = set()
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading corpus IDs"):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))

print("Loading qrels...")
qrels = {}
valid_queries = set()
with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading qrels"):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, docid, rel = parts[0], parts[2], int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print("Loading topics...")
topics = {}
with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])
qrels_df  = pd.DataFrame([
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
])

print(f"Topics: {len(topics_df)} | Qrels: {len(qrels_df)}\n")

# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")

hits_at_5     = 0
hits_at_10    = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    encode_ms = (time.time() - t0) * 1000

    # Weaviate search
    t1 = time.time()
    results = col.query.near_vector(
        near_vector=query_emb[0].tolist(),
        limit=50,
        return_metadata=MetadataQuery(distance=True),
        return_properties=["doc_id"]
    )
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    # Deduplicate chunks → unique docs
    retrieved_top10 = []
    seen = set()
    for r in results.objects:
        doc_id = str(r.properties["doc_id"])
        if doc_id not in seen:
            seen.add(doc_id)
            retrieved_top10.append(doc_id)
        if len(retrieved_top10) == 10:
            break

    retrieved_top5  = set(retrieved_top10[:5])
    retrieved_top10 = set(retrieved_top10)

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat  = np.array(query_latencies)
elat = np.array(encode_latencies)
slat = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "vector_store":      "Weaviate-Embedded",
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

# ===================== CLOSE CLIENT =====================
client.close()
print("\nWeaviate client closed.")

Loading model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.22k [00:00<?, ?B/s]

...

Loading qrels: 0it [00:00, ?it/s]

Loading topics...
Topics: 140 | Qrels: 174

Evaluating 140 queries...


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API


==================== RESULTS ====================
  Total queries:  140
  Hit@5:          0.6000
  Hit@10:         0.6929

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    39.56 ms
    Median:  35.83 ms
    P95:     69.85 ms
    P99:     82.15 ms

  Encode only:
    Mean:    27.66 ms
    Median:  24.93 ms

  Search only:
    Mean:    11.90 ms
    Median:  11.25 ms

     vector_store  chunk_size  chunk_overlap  total_docs  total_chunks  total_queries  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms  encode_mean_ms  search_mean_ms
Weaviate-Embedded         600             90       68611        165125            140    0.6  0.6929            39.56           69.85           82.15           27.66            11.9

Weaviate client closed.


In [ ]:
print(f"""
============== CHUNKING SUMMARY ==============
  Total docs:              {total_docs}
  Total chunks:            {len(all_chunks_text)}
  Avg chunks per doc:      {len(all_chunks_text)/total_docs:.1f}
  Min chunk length (char): {min(len(c) for c in all_chunks_text)}
  Max chunk length (char): {max(len(c) for c in all_chunks_text)}
  Avg chunk length (char): {sum(len(c) for c in all_chunks_text)/len(all_chunks_text):.1f}
==============================================
""")


============== CHUNKING SUMMARY ==============
  Total docs:              68611
  Total chunks:            165125
  Avg chunks per doc:      2.4
  Min chunk length (char): 1
  Max chunk length (char): 600
  Avg chunk length (char): 410.6



In [ ]:
!pip install -q pymilvus milvus-lite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.8/344.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


# Milvus

In [ ]:
from pymilvus import MilvusClient

# ===================== CONFIG =====================

db_path = "/content/milvus_miracl.db"   # local file — no server needed

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks_text = []
all_chunks_meta = []
total_docs = 0

print("Chunking documents...")
with open(corpus_file, 'r', encoding='utf-8') as f:
    for doc_idx, line in enumerate(tqdm(f, desc="Chunking")):
        if not line.strip():
            continue

        doc    = json.loads(line)
        text   = doc.get("text", "")
        doc_id = str(doc.get("docid") or doc.get("id") or doc_idx)

        if len(text) < 600:
            continue

        total_docs += 1
        chunks = splitter.split_text(text)

        for chunk_idx, chunk_text in enumerate(chunks):
            all_chunks_text.append(chunk_text)
            all_chunks_meta.append({
                "doc_id":              doc_id,
                "chunk_id":            chunk_idx,
                "original_doc_length": len(text)
            })

print(f"""
============== CHUNKING SUMMARY ==============
  Total docs:              {total_docs}
  Total chunks:            {len(all_chunks_text)}
  Avg chunks per doc:      {len(all_chunks_text)/total_docs:.1f}
  Min chunk length (char): {min(len(c) for c in all_chunks_text)}
  Max chunk length (char): {max(len(c) for c in all_chunks_text)}
  Avg chunk length (char): {sum(len(c) for c in all_chunks_text)/len(all_chunks_text):.1f}
==============================================
""")

# ===================== ENCODE =====================
print(f"Encoding {len(all_chunks_text)} chunks with batch_size={batch_size}...")
encode_start = time.time()

embeddings = model.encode(
    all_chunks_text,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')

encode_elapsed = time.time() - encode_start

print(f"""
============== ENCODING ==============
  Total time:        {encode_elapsed:.1f}s ({encode_elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks_text)/encode_elapsed:.1f}
  Embedding dim:     {embeddings.shape[1]}
  Matrix size:       {embeddings.nbytes/1024**2:.2f} MB
======================================
""")

# ===================== BUILD MILVUS INDEX =====================
print("Building Milvus index...")
index_start = time.time()

# Milvus Lite — local file, no server needed
client = MilvusClient(db_path)

# Drop collection if exists
if client.has_collection(collection):
    client.drop_collection(collection)
    print(f"Dropped existing collection: {collection}")

# Create collection
client.create_collection(
    collection_name=collection,
    dimension=embeddings.shape[1],
    metric_type="COSINE",
    auto_id=True
)
print(f"Created collection: {collection}")

# Upload in batches
upload_batch = 512
for i in tqdm(range(0, len(embeddings), upload_batch), desc="Uploading to Milvus"):
    batch_end = min(i + upload_batch, len(embeddings))
    data = [
        {
            "vector":              embeddings[j].tolist(),
            "doc_id":              all_chunks_meta[j]["doc_id"],
            "chunk_id":            all_chunks_meta[j]["chunk_id"],
            "original_doc_length": all_chunks_meta[j]["original_doc_length"]
        }
        for j in range(i, batch_end)
    ]
    client.insert(collection_name=collection, data=data)

index_elapsed = time.time() - index_start

print(f"""
============== MILVUS INDEX ==============
  Collection:    {collection}
  Total vectors: {client.get_collection_stats(collection)['row_count']}
  Build time:    {index_elapsed:.2f}s
==========================================
""")

# ===================== LOAD TOPICS & QRELS =====================
print("Loading filtered doc IDs...")
filtered_doc_ids = set()
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading corpus IDs"):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))

print("Loading qrels...")
qrels = {}
valid_queries = set()
with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading qrels"):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, docid, rel = parts[0], parts[2], int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print("Loading topics...")
topics = {}
with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])
qrels_df  = pd.DataFrame([
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
])

print(f"Topics: {len(topics_df)} | Qrels: {len(qrels_df)}\n")

# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")

hits_at_5     = 0
hits_at_10    = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    encode_ms = (time.time() - t0) * 1000

    # Milvus search
    t1 = time.time()
    results = client.search(
        collection_name=collection,
        data=[query_emb[0].tolist()],
        limit=50,
        output_fields=["doc_id"]
    )[0]
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    # Deduplicate chunks → unique docs
    retrieved_top10 = []
    seen = set()
    for r in results:
        doc_id = str(r["entity"]["doc_id"])
        if doc_id not in seen:
            seen.add(doc_id)
            retrieved_top10.append(doc_id)
        if len(retrieved_top10) == 10:
            break

    retrieved_top5  = set(retrieved_top10[:5])
    retrieved_top10 = set(retrieved_top10)

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat  = np.array(query_latencies)
elat = np.array(encode_latencies)
slat = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "vector_store":      "Milvus-Lite",
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

# ===================== CLOSE CLIENT =====================
client.close()
print("\nMilvus client closed.")

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Chunking documents...


...

Loading qrels: 0it [00:00, ?it/s]

Loading topics...
Topics: 140 | Qrels: 174

Evaluating 140 queries...


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API


==================== RESULTS ====================
  Total queries:  140
  Hit@5:          0.5714
  Hit@10:         0.6571

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    1138.19 ms
    Median:  826.04 ms
    P95:     3428.04 ms
    P99:     4167.99 ms

  Encode only:
    Mean:    37.95 ms
    Median:  30.92 ms

  Search only:
    Mean:    1100.23 ms
    Median:  793.79 ms

vector_store  chunk_size  chunk_overlap  total_docs  total_chunks  total_queries  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms  encode_mean_ms  search_mean_ms
 Milvus-Lite         600             90       68611        165125            140 0.5714  0.6571          1138.19         3428.04         4167.99           37.95         1100.23

Milvus client closed.


# jinaa3

In [ ]:
# ===================== CONFIG =====================
corpus_file   = "/content/long_docs.jsonl"
chunk_size    = 600
chunk_overlap = 90
batch_size    = 64
model_name    = "jinaai/jina-embeddings-v3"
qrels_file    = "/content/qrels.miracl-v1.0-fa-dev.tsv"
topics_file   = "/content/topics.miracl-v1.0-fa-dev.tsv"
collection    = "Miracl"